# Task 7: Understanding User Preferences and Preparing Interaction Data

## Objective
So far my recommender only looks at the text of TED Talks (TF-IDF and cosine similarity), so it gives the same recommendations to everyone. A real recommendation system should also learn from people: what they watched, what they liked and what they skipped. In this notebook I learn what that kind of data looks like, build a sample interaction dataset, turn it into a user-item matrix, analyse it, and understand the idea of collaborative filtering.

> **Important: all user data in this notebook is SIMULATED.** I do not have real user data for this project, so I generated 24 made-up users using a fixed random seed. The TED Talks are real (from my cleaned dataset), but the users, ratings, likes and bookmarks are invented for learning. Every row in the CSV files has a `data_source` column that says "simulated".

## Key concepts in plain words
Think of a restaurant. Users are customers and TED Talks are dishes on the menu.

| Concept | What it means | Restaurant picture |
|---|---|---|
| User ID | A unique number for each person | A customer's membership number |
| TED Talk ID | A unique number for each talk (`talk_id` in my dataset) | The dish number on the menu |
| Rating | A score from 1 to 5 given by a user to a talk | Star rating for a dish |
| Like / dislike | A quick reaction. In my data: rating 4-5 = like, 3 = neutral, 1-2 = dislike | Thumbs up or thumbs down |
| Watch history | The list of talks a user has watched | What the customer ordered before |
| Saved / bookmarked | Talks a user marked to watch later | A wishlist of dishes to try |
| User preferences | Interests a user tells us, for example when signing up | "I am vegetarian" |
| User-item interaction matrix | A table with users as rows, talks as columns and ratings as values. Blank means no rating | A big order sheet with customers down the side and dishes across the top |

**Two kinds of feedback:** ratings, likes and dislikes are *explicit* (the user tells us directly). Watching and bookmarking are *implicit* (we infer interest from behaviour).

## How interactions reveal interests
If a user gives 5 stars to three climate talks and 1 star to two finance talks, the system learns what that person enjoys without reading a single word of any transcript. When many users leave this kind of trail, the system can also spot people with similar taste and use that to recommend talks. That is the idea behind collaborative filtering, which I explain later in this notebook.

In [14]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)   # so my 24-column matrix is not cut off with "..."

In [15]:
# only the four columns I need, so loading is quick
df = pd.read_csv("../data/processed/ted_talks_cleaned.csv",
                 usecols=["talk_id", "title", "topics", "views"])
print(df.shape)
df.head(3)

(4005, 4)


,talk_id,title,views,topics
0,1,Averting the climate crisis,3523392,"['alternative energy', 'cars', 'climate change', 'cultur..."
1,92,The best stats you've ever seen,14501685,"['Africa', 'Asia', 'Google', 'demo', 'economics', 'globa..."
2,7,Simplicity sells,1920832,"['computers', 'entertainment', 'interface design', 'medi..."


## Designing the simulated dataset

**What I built**
- **24 real TED Talks:** 6 each from four interests (Technology, Education, Health, Environment). For each interest I took the most-viewed talks with that topic tag, so they are talks people would plausibly watch.
- **24 simulated users:** 6 for each favourite interest. The favourite interest is what a user would choose at sign-up. It is saved in `user_preferences.csv`.
- I kept it small (24 users x 24 talks) so the whole matrix fits on one screen.

**Rules I used to simulate behaviour (simple but realistic)**
1. People mostly watch talks from their favourite interest and only sometimes explore other topics.
2. For a favourite-interest talk, users usually give 4 or 5. For other topics ratings are lower, mostly 1 to 3.
3. Some users are heavy watchers and some are light watchers.
4. People bookmark talks they loved more often than talks they did not like. A few talks are bookmarked without being watched.
5. A fixed random seed (42) gives me the same data every time I run the notebook.

**Files created**

| File | What it holds |
|---|---|
| `data/interaction/user_interactions.csv` | One row per user-talk interaction: user_id, talk_id, watched, rating, reaction, saved, data_source |
| `data/interaction/user_preferences.csv` | One row per user: user_id, favorite_interest, data_source |

**Limitation:** because I wrote the rules, the pattern "people like their favourite topics" is built into this data. It is not discovered from real behaviour, so it is good for learning the method, not for proving anything about real users.

In [16]:
# four interests, and the topic keyword that defines each one
groups = {
    "Technology": "technology",
    "Education": "education",
    "Health": "health",
    "Environment": "environment",
}
talks_per_group = 6

catalog_rows = []
used_ids = set()          # so one talk is never picked for two groups

for group_name, keyword in groups.items():
    pool = df[df["topics"].str.contains(keyword, case=False, na=False)]
    pool = pool[~pool["talk_id"].isin(used_ids)]
    top = pool.sort_values("views", ascending=False).head(talks_per_group)
    used_ids.update(top["talk_id"])
    for _, row in top.iterrows():
        catalog_rows.append({
            "talk_id": int(row["talk_id"]),
            "title": row["title"],
            "interest_group": group_name,
        })

talk_catalog = pd.DataFrame(catalog_rows)
talk_titles = talk_catalog.set_index("talk_id")["title"]   # quick lookup: talk_id -> title

print(talk_catalog.shape)
display(talk_catalog)

(24, 3)


,talk_id,title,interest_group
0,2405,This is what happens when you reply to spam email,Technology
1,2577,The agony of trying to unsubscribe,Technology
2,685,The thrilling potential of SixthSense technology,Technology
3,2217,The price of shame,Technology
4,2774,The future we're building -- and boring,Technology
5,206,Underwater astonishments,Technology
6,66,Do schools kill creativity?,Education
7,1733,Grit: The power of passion and perseverance,Education
8,1386,Questions no one knows the answers to,Education
9,24138,The language of lying,Education


In [17]:
users_per_group = 6

user_rows = []
next_user_id = 1
for group_name in groups:
    for _ in range(users_per_group):
        user_rows.append({"user_id": next_user_id, "favorite_interest": group_name})
        next_user_id += 1

user_preferences = pd.DataFrame(user_rows)
user_preferences["data_source"] = "simulated"
user_preferences.head(8)

,user_id,favorite_interest,data_source
0,1,Technology,simulated
1,2,Technology,simulated
2,3,Technology,simulated
3,4,Technology,simulated
4,5,Technology,simulated
5,6,Technology,simulated
6,7,Education,simulated
7,8,Education,simulated


In [18]:
rng = np.random.default_rng(42)     # fixed seed: same simulated data every time

def rating_to_reaction(rating):
    if rating >= 4:
        return "like"
    if rating <= 2:
        return "dislike"
    return "neutral"

records = []
for _, user in user_preferences.iterrows():
    activity = rng.uniform(0.5, 1.2)      # some users watch a lot, some only a little

    for _, talk in talk_catalog.iterrows():
        is_favorite = talk["interest_group"] == user["favorite_interest"]
        watch_chance = min((0.75 if is_favorite else 0.20) * activity, 0.95)

        if rng.random() < watch_chance:
            # the user watched it, so they also rate it
            if is_favorite:
                rating = int(rng.choice([3, 4, 5], p=[0.10, 0.40, 0.50]))
            else:
                rating = int(rng.choice([1, 2, 3, 4], p=[0.25, 0.35, 0.30, 0.10]))
            save_chance = 0.40 if rating >= 4 else 0.05      # loved talks get bookmarked more
            records.append({
                "user_id": int(user["user_id"]),
                "talk_id": int(talk["talk_id"]),
                "watched": 1,
                "rating": rating,
                "reaction": rating_to_reaction(rating),
                "saved": int(rng.random() < save_chance),
            })
        elif rng.random() < 0.05:
            # not watched, only bookmarked for later (so no rating)
            records.append({
                "user_id": int(user["user_id"]),
                "talk_id": int(talk["talk_id"]),
                "watched": 0,
                "rating": np.nan,
                "reaction": "none",
                "saved": 1,
            })

interactions = pd.DataFrame(records)
interactions["data_source"] = "simulated"
print("Total interaction rows:", len(interactions))
interactions.head(10)

Total interaction rows: 181


,user_id,talk_id,watched,rating,reaction,saved,data_source
0,1,2405,1,5.0,like,0,simulated
1,1,2577,1,5.0,like,0,simulated
2,1,2217,1,4.0,like,0,simulated
3,1,2774,1,5.0,like,0,simulated
4,1,206,1,5.0,like,1,simulated
5,1,1728,0,NaN,none,1,simulated
6,1,1272,1,3.0,neutral,0,simulated
7,1,1815,1,1.0,dislike,0,simulated
8,1,3136,1,1.0,dislike,0,simulated
9,2,2405,1,4.0,like,0,simulated


In [19]:
os.makedirs("../data/interaction", exist_ok=True)     # creates the folder if it doesn't exist yet
interactions.to_csv("../data/interaction/user_interactions.csv", index=False)
user_preferences.to_csv("../data/interaction/user_preferences.csv", index=False)

print("Rows with a rating:", int(interactions["rating"].notna().sum()))
print("Bookmark-only rows (not watched, no rating):", int((interactions["watched"] == 0).sum()))
print("Saved to: data/interaction/")

Rows with a rating: 164
Bookmark-only rows (not watched, no rating): 17
Saved to: data/interaction/


## User-item interaction matrix
Now I turn the ratings into one table:
- **Rows** = users
- **Columns** = TED Talks (by `talk_id`)
- **Values** = the rating the user gave
- **Blank (NaN)** = the user has not rated that talk

A blank does **not** mean "disliked". It means "no information". Only ratings go into the matrix, so bookmark-only rows also show as blank. Most cells will be blank, which is normal: real recommendation data is almost always like this, and it is called *sparse*.

In [20]:
rated = interactions.dropna(subset=["rating"])      # only rows that have a rating
user_item_matrix = rated.pivot(index="user_id", columns="talk_id", values="rating")

filled = int(user_item_matrix.notna().sum().sum())
total_cells = user_item_matrix.size

print("Matrix shape (users x talks):", user_item_matrix.shape)
print("Filled cells:", filled, "| Empty cells:", total_cells - filled)
print(f"Share of the matrix that is empty: {100 * (total_cells - filled) / total_cells:.1f}%")
user_item_matrix

Matrix shape (users x talks): (24, 24)
Filled cells: 164 | Empty cells: 412
Share of the matrix that is empty: 71.5%


talk_id,66,206,263,685,1042,1272,1344,1386,1728,1733,1815,2217,2361,2399,2405,2452,2476,2558,2577,2774,3136,24138,24147,32560
user_id,,,,,,,,,,,,,,,,,,,,,,,,
1,NaN,5.0,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,1.0,4.0,NaN,NaN,5.0,NaN,NaN,NaN,5.0,5.0,1.0,NaN,NaN,NaN
2,NaN,3.0,NaN,5.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,3.0,NaN,NaN,4.0,NaN,NaN,NaN,5.0,4.0,4.0,NaN,1.0,NaN
3,NaN,NaN,2.0,4.0,NaN,NaN,2.0,NaN,NaN,4.0,NaN,4.0,NaN,NaN,5.0,NaN,NaN,NaN,5.0,5.0,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,4.0,3.0,NaN,NaN
5,3.0,5.0,1.0,4.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN
6,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,4.0,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,1.0,5.0,NaN,4.0,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN
8,5.0,NaN,NaN,2.0,NaN,4.0,NaN,NaN,4.0,5.0,NaN,NaN,NaN,1.0,3.0,NaN,NaN,NaN,2.0,NaN,NaN,3.0,NaN,1.0
9,4.0,NaN,NaN,NaN,NaN,4.0,2.0,5.0,4.0,5.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,2.0,NaN


## Basic interaction data analysis
In this section I answer these questions from the data:
1. How many unique users are there?
2. How many unique TED Talks are included?
3. Which talks got the highest ratings?
4. Which users rated the most talks?
5. Which talks got the most interactions (watched or bookmarked)?
6. What does a missing rating mean?

I also add one extra check: do users rate talks from their favourite interest higher? (This will be true by design, because I wrote the simulation rules.)

In [21]:
print("1) Unique users:", interactions["user_id"].nunique())
print("2) Unique TED Talks:", interactions["talk_id"].nunique())

# 3) highest rated talks (only talks with at least 3 ratings, so one lucky 5 doesn't win)
talk_stats = rated.groupby("talk_id")["rating"].agg(avg_rating="mean", num_ratings="count")
talk_stats = talk_stats.join(talk_titles)
best = talk_stats[talk_stats["num_ratings"] >= 3].sort_values("avg_rating", ascending=False).head(5)
print("\n3) Highest rated talks (at least 3 ratings):")
display(best[["title", "avg_rating", "num_ratings"]].round(2))

# 4) users who rated the most talks
ratings_per_user = rated.groupby("user_id")["rating"].count().sort_values(ascending=False)
print("\n4) Users who rated the most talks:")
display(ratings_per_user.head(5).rename("num_ratings").to_frame())

# 5) talks with the most interactions (watched or bookmarked)
interactions_per_talk = interactions.groupby("talk_id").size().sort_values(ascending=False).head(5)
most_interacted = interactions_per_talk.rename("num_interactions").to_frame().join(talk_titles)
print("\n5) Talks with the most interactions:")
display(most_interacted[["title", "num_interactions"]])

# 6) missing ratings
print("\n6) Empty cells in the matrix:", int(user_item_matrix.isna().sum().sum()))
print("   A missing rating means the user has not rated that talk. It does NOT mean they disliked it.")

# extra check: do users rate their favourite interest higher?
check = (rated
         .merge(user_preferences[["user_id", "favorite_interest"]], on="user_id")
         .merge(talk_catalog[["talk_id", "interest_group"]], on="talk_id"))
check["talk_in_favorite_interest"] = check["interest_group"] == check["favorite_interest"]
print("\nExtra) Average rating: favourite-interest talks (True) vs other talks (False):")
display(check.groupby("talk_in_favorite_interest")["rating"]
        .agg(avg_rating="mean", num_ratings="count").round(2))

1) Unique users: 24
2) Unique TED Talks: 24

3) Highest rated talks (at least 3 ratings):


,title,avg_rating,num_ratings
talk_id,,,
1386,Questions no one knows the answers to,4.5,4
2452,This country isn't just carbon neutral -- it's carbon ne...,4.2,5
2405,This is what happens when you reply to spam email,4.0,6
2577,The agony of trying to unsubscribe,4.0,8
2558,How trees talk to each other,4.0,5



4) Users who rated the most talks:


,num_ratings
user_id,
18,11
13,11
8,10
10,10
9,9



5) Talks with the most interactions:


,title,num_interactions
talk_id,,
263,What's wrong with what we eat,10
1272,A 12-year-old app developer,10
2577,The agony of trying to unsubscribe,10
2399,What makes a good life? Lessons from the longest study o...,10
1728,Every kid needs a champion,9



6) Empty cells in the matrix: 412
   A missing rating means the user has not rated that talk. It does NOT mean they disliked it.

Extra) Average rating: favourite-interest talks (True) vs other talks (False):


,avg_rating,num_ratings
talk_in_favorite_interest,,
False,2.04,70
True,4.36,94


## Collaborative filtering: the basic idea

**Content-based filtering** (my Tasks 4 to 6) says: "you liked this talk, so here are talks with similar text."
**Collaborative filtering** says: "people with similar taste to yours liked these talks." It never reads the text at all. It only uses the ratings table.

**Everyday picture:** you and a friend have loved the same five movies. She watches a new one and loves it. You trust her pick, even if you know nothing about that movie.

**With TED Talks**
1. User A and User B rate several talks in a similar way, so the system decides they have similar taste.
2. User A loved a talk that User B has not watched.
3. The system recommends that talk to User B.

**Two flavours:** *user-based* (find similar users) and *item-based* (find talks that the same people liked together).

**A known problem:** a brand-new user has no ratings yet, so the system has no one to compare them with. This is called the *cold start* problem. It is one reason why real systems combine collaborative and content-based filtering.

**Demo below:** I pick the most active user as User A, find the user who rated the most of the same talks (User B), put their ratings side by side, and list talks that User A loved but User B has not rated.

In [22]:
# User A = the user with the most ratings
user_a = rated.groupby("user_id")["rating"].count().idxmax()

# User B = the user who rated the most of the same talks as User A
seen = user_item_matrix.notna().astype(int)                 # 1 = rated, 0 = not rated
shared_with_a = seen.drop(user_a).dot(seen.loc[user_a])     # how many talks each other user shares with A
user_b = shared_with_a.idxmax()

pair = user_item_matrix.loc[[user_a, user_b]].T             # rows = talks, columns = the two users
pair.columns = ["User A", "User B"]
pair = pair.join(talk_titles)

print(f"User A = user {user_a}, User B = user {user_b}")
print("Their favourite interests:",
      user_preferences.set_index("user_id").loc[[user_a, user_b], "favorite_interest"].tolist())

print("\nTalks both users rated (side by side):")
shared = pair.dropna(subset=["User A", "User B"])
display(shared)

suggestions = pair[(pair["User A"] >= 4) & (pair["User B"].isna())]
print("\nTalks User A rated 4 or 5 that User B has not rated (suggestions for User B):")
if suggestions.empty:
    print("None for this pair.")
else:
    display(suggestions)

User A = user 13, User B = user 18
Their favourite interests: ['Health', 'Health']

Talks both users rated (side by side):


,User A,User B,title
talk_id,,,
1042,4.0,4.0,The power of vulnerability
1344,4.0,4.0,The happy secret to better work
2399,5.0,4.0,What makes a good life? Lessons from the longest study o...
3136,1.0,4.0,What's hidden under the Greenland ice sheet?
24138,2.0,2.0,The language of lying
24147,5.0,3.0,What are those floaty things in your eye?



Talks User A rated 4 or 5 that User B has not rated (suggestions for User B):


,User A,User B,title
talk_id,,,
263,4.0,NaN,What's wrong with what we eat
1815,5.0,NaN,How to make stress your friend
2361,4.0,NaN,How to stay calm when you know you'll be stressed


In [23]:
from IPython.display import Markdown, display

# Observations written from the real numbers in my data (so they always match the outputs above)
n_users = interactions["user_id"].nunique()
n_talks = interactions["talk_id"].nunique()
n_ratings = len(rated)
empty_cells = total_cells - filled
pct_empty = 100 * empty_cells / total_cells
bookmark_only = int((interactions["watched"] == 0).sum())

top_talk = best.iloc[0] if len(best) else talk_stats.sort_values("avg_rating", ascending=False).iloc[0]
busiest_user = int(ratings_per_user.index[0])
busiest_count = int(ratings_per_user.iloc[0])
popular = most_interacted.iloc[0]

# ties: several users or talks can share the same top number
all_counts = interactions.groupby("talk_id").size()
talk_ties = int((all_counts == all_counts.max()).sum()) - 1
user_ties = int((ratings_per_user == ratings_per_user.max()).sum()) - 1
user_tie_text = f", tied with {user_ties} other user{'s' if user_ties != 1 else ''}" if user_ties > 0 else ""
talk_tie_text = f", tied with {talk_ties} other talk{'s' if talk_ties != 1 else ''}" if talk_ties > 0 else ""

avg_by_group = check.groupby("talk_in_favorite_interest")["rating"].mean()
fav_avg, other_avg = avg_by_group.loc[True], avg_by_group.loc[False]

interest_a, interest_b = user_preferences.set_index("user_id").loc[[user_a, user_b], "favorite_interest"].tolist()
shared_count = len(shared)
gap_text = ""
if shared_count > 0:
    gap = (shared["User A"] - shared["User B"]).abs().mean()
    gap_text = f" On those talks their ratings differed by {gap:.2f} stars on average (the smaller this number, the more similar their taste)."

if suggestions.empty:
    suggestion_text = (f"User {user_a} has no 4 or 5 rated talk that user {user_b} has missed, "
                       f"so this pair gives no suggestion. A bigger dataset would give more.")
else:
    title_list = "; ".join(f'"{t}"' for t in suggestions["title"].head(3))
    suggestion_text = (f"User {user_a} rated these 4 or 5 and user {user_b} has not rated them yet: {title_list}. "
                       f"A collaborative filtering system would suggest them to user {user_b}.")

report_sections = [
    ("Size of the data",
     f"The simulated data has {n_users} users, {n_talks} TED Talks and {n_ratings} ratings. "
     f"The matrix has {filled} filled cells and {empty_cells} empty cells, so {pct_empty:.0f}% of it is empty. "
     f"This is normal for recommendation data, because nobody watches everything."),
    ("Ratings and popularity",
     f"The highest rated talk (with at least 3 ratings) is \"{top_talk['title']}\" with an average of "
     f"{top_talk['avg_rating']:.2f} from {int(top_talk['num_ratings'])} ratings. "
     f"The most active user is user {busiest_user}, who rated {busiest_count} talks{user_tie_text}. "
     f"The talk with the most interactions is \"{popular['title']}\" with {int(popular['num_interactions'])} "
     f"interactions (watched or bookmarked){talk_tie_text}."),
    ("Missing ratings",
     f"An empty cell means the user has not rated that talk. It does not mean they disliked it, so I never treat a "
     f"blank as a low rating. {bookmark_only} rows are bookmark-only (saved but not watched), and they also show as "
     f"blank because there is no rating to put in the matrix."),
    ("Preferences shape ratings",
     f"Users rated talks from their favourite interest {fav_avg:.2f} on average, compared with {other_avg:.2f} for "
     f"talks from other interests. I built this pattern into the simulation, so it is not a discovery. But it shows "
     f"what real data would reveal: ratings reflect personal taste, so the same talk should not be recommended to "
     f"everyone in the same way."),
    ("Collaborative filtering example",
     f"User {user_a} (favourite interest: {interest_a}) and user {user_b} (favourite interest: {interest_b}) both "
     f"rated {shared_count} of the same talks.{gap_text} {suggestion_text}"),
    ("How this helps personalised recommendations",
     "My text-based recommender from Tasks 4 to 6 gives every user the same list for the same talk. With interaction "
     "data the system can (1) skip talks a user has already watched, (2) learn each user's favourite topics, "
     "(3) recommend talks that similar users loved even when the text of the talks is different, and (4) combine "
     "both approaches, so that a brand-new user with no ratings still gets content-based recommendations until "
     "enough ratings are collected. The next step is to build the collaborative filtering model on this matrix."),
]

display(Markdown("## Observations (calculated from my simulated data)"))
for heading, text in report_sections:
    display(Markdown(f"**{heading}.** {text}"))


## Observations (calculated from my simulated data)

**Size of the data.** The simulated data has 24 users, 24 TED Talks and 164 ratings. The matrix has 164 filled cells and 412 empty cells, so 72% of it is empty. This is normal for recommendation data, because nobody watches everything.

**Ratings and popularity.** The highest rated talk (with at least 3 ratings) is "Questions no one knows the answers to" with an average of 4.50 from 4 ratings. The most active user is user 18, who rated 11 talks, tied with 1 other user. The talk with the most interactions is "What's wrong with what we eat" with 10 interactions (watched or bookmarked), tied with 3 other talks.

**Missing ratings.** An empty cell means the user has not rated that talk. It does not mean they disliked it, so I never treat a blank as a low rating. 17 rows are bookmark-only (saved but not watched), and they also show as blank because there is no rating to put in the matrix.

**Preferences shape ratings.** Users rated talks from their favourite interest 4.36 on average, compared with 2.04 for talks from other interests. I built this pattern into the simulation, so it is not a discovery. But it shows what real data would reveal: ratings reflect personal taste, so the same talk should not be recommended to everyone in the same way.

**Collaborative filtering example.** User 13 (favourite interest: Health) and user 18 (favourite interest: Health) both rated 6 of the same talks. On those talks their ratings differed by 1.00 stars on average (the smaller this number, the more similar their taste). User 13 rated these 4 or 5 and user 18 has not rated them yet: "What's wrong with what we eat"; "How to make stress your friend"; "How to stay calm when you know you'll be stressed". A collaborative filtering system would suggest them to user 18.

**How this helps personalised recommendations.** My text-based recommender from Tasks 4 to 6 gives every user the same list for the same talk. With interaction data the system can (1) skip talks a user has already watched, (2) learn each user's favourite topics, (3) recommend talks that similar users loved even when the text of the talks is different, and (4) combine both approaches, so that a brand-new user with no ratings still gets content-based recommendations until enough ratings are collected. The next step is to build the collaborative filtering model on this matrix.

## Limitations and what comes next

- **All user data is simulated.** I wrote the rules myself (people mostly like their favourite interest), so that pattern is built in. It is not discovered from real behaviour. This data is good for learning the method, not for proving anything about real users.
- **Interest groups are based on TED's topic tags, which are broad.** A few talks fit their group loosely, for example "Underwater astonishments" under Technology and "The language of lying" under Education.
- **The data is small and sparse.** 24 users and 24 talks, with about 70% of the matrix empty. Real systems have thousands of users and are even emptier.
- **Cold start.** A new user with no ratings cannot be compared with anyone, so collaborative filtering alone cannot help them. That is why real systems combine it with the content-based recommender I built in Tasks 4 to 6.

**Next step:** use this interaction matrix to build a collaborative filtering model that recommends talks to a user based on similar users.


In [24]:
print(sorted(os.listdir("../data/interaction")))
pd.read_csv("../data/interaction/user_interactions.csv").head()


['user_interactions.csv', 'user_preferences.csv']


,user_id,talk_id,watched,rating,reaction,saved,data_source
0,1,2405,1,5.0,like,0,simulated
1,1,2577,1,5.0,like,0,simulated
2,1,2217,1,4.0,like,0,simulated
3,1,2774,1,5.0,like,0,simulated
4,1,206,1,5.0,like,1,simulated


In [25]:
%pip install reportlab


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
from xml.sax.saxutils import escape
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak

styles = getSampleStyleSheet()
body = ParagraphStyle("body", parent=styles["BodyText"], fontSize=10, leading=14)
h1 = ParagraphStyle("h1", parent=styles["Heading1"], fontSize=18)
h2 = ParagraphStyle("h2", parent=styles["Heading2"], fontSize=13)
warning = ParagraphStyle("warning", parent=body, backColor=colors.HexColor("#FFF4CE"), borderPadding=6)

def make_table(frame, widths=None, font=8):
    cell = ParagraphStyle("cell", parent=body, fontSize=font, leading=font + 2)
    header = [frame.index.name or ""] + [str(c) for c in frame.columns]
    rows = [[Paragraph(escape(h), cell) for h in header]]
    for idx, row in frame.iterrows():
        cells = [Paragraph(escape(str(idx)), cell)]
        for v in row:
            cells.append(Paragraph("" if pd.isna(v) else escape(str(v)), cell))
        rows.append(cells)
    table = Table(rows, colWidths=widths, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#E8E8E8")),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ]))
    return table

story = []
story.append(Paragraph("Task 7: Understanding User Preferences and Preparing Interaction Data", h1))
story.append(Paragraph("Observations report", h2))
story.append(Paragraph(
    "<b>Important: all user data in this report is SIMULATED.</b> I do not have real user data for this project, so I "
    "generated 24 made-up users with a fixed random seed (42). The 24 TED Talks are real and come from my cleaned "
    "dataset. Every row in the CSV files has a data_source column that says \"simulated\".", warning))
story.append(Spacer(1, 10))

story.append(Paragraph("Key concepts", h2))
concepts = pd.DataFrame({
    "Meaning": [
        "A unique number for each person.",
        "A unique number for each talk (talk_id in my dataset).",
        "A score from 1 to 5 that a user gives to a talk.",
        "A quick reaction. In my data: rating 4-5 = like, 3 = neutral, 1-2 = dislike.",
        "The talks a user has watched.",
        "Talks a user saved to watch later.",
        "Interests a user chooses, for example when signing up.",
        "A table with users as rows, talks as columns and ratings as values. Blank = not rated.",
    ]
}, index=pd.Index(["User ID", "TED Talk ID", "Rating", "Like / dislike", "Watch history",
                   "Saved / bookmarked", "User preferences", "User-item matrix"], name="Concept"))
story.append(make_table(concepts, widths=[150, 600], font=9))

story.append(Paragraph("Dataset I created", h2))
story.append(Paragraph(
    f"{n_users} simulated users (6 for each favourite interest: Technology, Education, Health, Environment) and "
    f"{n_talks} real TED Talks (6 per interest group, chosen from the most-viewed talks with that topic tag). "
    f"{len(interactions)} interaction rows were saved to data/interaction/user_interactions.csv, and each user's "
    f"favourite interest to data/interaction/user_preferences.csv.", body))

story.append(PageBreak())
story.append(Paragraph("User-item interaction matrix (rows = users, columns = talk_id, blank = not rated)", h2))
matrix_view = user_item_matrix.astype("Int64")
story.append(make_table(matrix_view, widths=[34] + [31] * matrix_view.shape[1], font=6))

story.append(PageBreak())
story.append(Paragraph("Basic interaction data analysis", h2))
story.append(Paragraph(f"1) Unique users: <b>{n_users}</b>. 2) Unique TED Talks: <b>{n_talks}</b>. "
                       f"6) Empty cells in the matrix: <b>{empty_cells}</b> ({pct_empty:.0f}%). A missing rating means the "
                       f"user has not rated that talk. It does not mean they disliked it.", body))
story.append(Spacer(1, 6))
story.append(Paragraph("3) Highest rated talks (at least 3 ratings)", body))
story.append(make_table(best[["title", "avg_rating", "num_ratings"]].round(2), widths=[60, 450, 80, 80]))
story.append(Spacer(1, 6))
story.append(Paragraph("4) Users who rated the most talks", body))
story.append(make_table(ratings_per_user.head(5).rename("num_ratings").to_frame(), widths=[80, 100]))
story.append(Spacer(1, 6))
story.append(Paragraph("5) Talks with the most interactions (watched or bookmarked)", body))
story.append(make_table(most_interacted[["title", "num_interactions"]], widths=[60, 450, 100]))
story.append(Spacer(1, 6))
story.append(Paragraph("Extra check: average rating for favourite-interest talks (True) and other talks (False)", body))
story.append(make_table(check.groupby("talk_in_favorite_interest")["rating"]
                        .agg(avg_rating="mean", num_ratings="count").round(2), widths=[160, 90, 90]))

story.append(PageBreak())
story.append(Paragraph("Collaborative filtering", h2))
story.append(Paragraph(
    "Content-based filtering says: you liked this talk, so here are talks with similar text. Collaborative filtering "
    "says: people with similar taste to yours liked these talks. It never reads the text; it only uses the ratings "
    "table. If User A and User B rate several talks similarly, and User A loved a talk that User B has not watched, "
    "the system recommends that talk to User B. A known problem is cold start: a brand-new user has no ratings, so "
    "there is nobody to compare them with.", body))
story.append(Spacer(1, 6))
story.append(Paragraph(f"Demo: User A = user {user_a}, User B = user {user_b}. Talks both rated:", body))
story.append(make_table(shared, widths=[60, 60, 60, 450]))
story.append(Spacer(1, 6))
story.append(Paragraph("Talks User A rated 4 or 5 that User B has not rated (suggestions for User B):", body))
if suggestions.empty:
    story.append(Paragraph("None for this pair.", body))
else:
    story.append(make_table(suggestions, widths=[60, 60, 60, 450]))

story.append(PageBreak())
story.append(Paragraph("Observations", h2))
for heading, text in report_sections:
    story.append(Paragraph(f"<b>{escape(heading)}.</b> {escape(text)}", body))
    story.append(Spacer(1, 6))
story.append(Paragraph("Limitations", h2))
story.append(Paragraph(
    "All users, ratings, likes and bookmarks are simulated, and I wrote the rules (people mostly like their favourite "
    "interest), so the pattern is built in and not discovered. The interest groups come from TED's topic tags, which are "
    "broad, so a few talks fit their group loosely (for example Underwater astonishments under Technology and The "
    "language of lying under Education). With only 24 users and 24 talks, results are for learning the method, not for "
    "proving anything about real users.", body))

os.makedirs("../reports", exist_ok=True)
doc = SimpleDocTemplate("../reports/task_07_observations.pdf", pagesize=landscape(A4),
                        leftMargin=25, rightMargin=25, topMargin=25, bottomMargin=25)
doc.build(story)
print("PDF saved: reports/task_07_observations.pdf")


PDF saved: reports/task_07_observations.pdf
